# 01 Harmonização dos dados TIC Domicílios (2023-2025)

Este notebook pega os 6 arquivos brutos da CETIC (domicílios e indivíduos de 2023, 2024 e 2025), deixa os três anos com os mesmos nomes e os mesmos códigos, limpa os códigos de "não sabe / não respondeu / não se aplica", junta tudo e salva o resultado em `data/processed/`.

- **Entra:** os CSVs de `data/raw/individuos/` e `data/raw/domicilios/` (ver `data/raw/README.md`)
- **Sai:** 4 arquivos em `data/processed/` (ver `data/processed/README.md`)

**Como ler:** cada seção começa com um texto explicando o porquê, e cada célula de código tem comentários (as linhas que começam com `#`) explicando o que cada linha faz. Rodar sempre com **Restart + Run All**, de cima pra baixo.

**Roteiro**
1. Preparar o ambiente
2. Carregar os dados brutos
3. Dois sistemas de peso (achado importante)
4. Harmonizar os nomes das colunas
5. Marcar o ano
6. Conferência: o que o questionário filtra
7. Recodificar G1 e G2
8. Limpar os códigos de não resposta
9. Velocidade e tipo de conexão
10. Escolher as colunas
11. Empilhar os anos e juntar os módulos
12. Testes de integridade
13. Salvar

## 1. Preparar o ambiente

O notebook precisa de duas coisas antes de tudo: as bibliotecas e o endereço da pasta do projeto.

- `pandas` trabalha com tabelas. Uma tabela inteira é um **DataFrame** e uma coluna sozinha é uma **Series**.
- `numpy` dá o `np.nan`, que é o jeito de escrever "valor faltante".
- `os` monta caminhos de arquivo e `sys` diz ao Python onde procurar os nossos próprios módulos.
- A função `encontrar_raiz_projeto` resolve um problema do VS Code: às vezes ele roda o notebook a partir da pasta `notebooks/`, às vezes da raiz do repositório. Em vez de chutar um caminho fixo como `'../data'`, ela sobe pasta por pasta até achar `src/harmonizacao.py`.
- As funções de recodificação ficam em `src/harmonizacao.py`, pra poderem ser usadas também nos notebooks 02 e 03. Se alguém mudar esse arquivo, precisa reiniciar o kernel (Restart) pra mudança valer.

In [1]:
# Bibliotecas
import os                # monta caminhos de arquivo (pastas e nomes)
import sys               # controla onde o Python procura módulos pra importar
import numpy as np       # np.nan = "valor faltante"
import pandas as pd      # tabelas (DataFrame) e colunas (Series)


def encontrar_raiz_projeto(pasta_atual):
    # Sobe de pasta em pasta até achar a que tem src/harmonizacao.py
    pasta = os.path.abspath(pasta_atual)          # caminho completo da pasta onde o notebook está rodando
    while True:                                   # repete até achar (return) ou desistir (raise)
        if os.path.exists(os.path.join(pasta, 'src', 'harmonizacao.py')):
            return pasta                          # achou: essa é a raiz do projeto
        pasta_pai = os.path.dirname(pasta)        # a pasta "de cima"
        if pasta_pai == pasta:                    # não dá mais pra subir: chegou na raiz do disco
            raise FileNotFoundError('Não achei src/harmonizacao.py subindo a partir de ' + pasta_atual)
        pasta = pasta_pai                         # continua procurando uma pasta acima


raiz_projeto = encontrar_raiz_projeto(os.getcwd())   # os.getcwd() = pasta atual
if raiz_projeto not in sys.path:                     # sys.path = lista de pastas onde o Python procura módulos
    sys.path.append(raiz_projeto)                    # sem isso, o "from src..." abaixo não funciona

# Nossas funções, que estão em src/harmonizacao.py
from src.harmonizacao import (
    recodificar_g2,
    recodificar_velocidade,
    renomear_colunas,
    padronizar_pesos_replica,
)

print('Raiz do projeto:', raiz_projeto)

Raiz do projeto: c:\Users\pedro\OneDrive\Área de Trabalho\Projeto-codigo\apartheid-digital


## 2. Carregar os dados brutos

- Os CSVs da CETIC usam `;` pra separar as colunas e `,` como decimal (formato brasileiro). Sem o `decimal=','`, um peso como `1234,56` seria lido como texto.
- Cada arquivo vira um DataFrame com um nome: `ind_` = indivíduos e `dom_` = domicílios.
- No fim, a célula guarda quantas linhas tinha cada módulo. Esses totais são usados nos testes da seção 12, pra garantir que nenhuma linha se perdeu nem se duplicou no caminho.

In [2]:
# Pastas onde ficam os CSVs brutos
pasta_ind = os.path.join(raiz_projeto, 'data', 'raw', 'individuos')
pasta_dom = os.path.join(raiz_projeto, 'data', 'raw', 'domicilios')

# Lê os 6 arquivos. sep=';' separa as colunas e decimal=',' lê "1234,56" como número
ind_2023 = pd.read_csv(os.path.join(pasta_ind, '2023_individuos.csv'), sep=';', decimal=',')
ind_2024 = pd.read_csv(os.path.join(pasta_ind, '2024_individuos.csv'), sep=';', decimal=',')
ind_2025 = pd.read_csv(os.path.join(pasta_ind, '2025_individuos.csv'), sep=';', decimal=',')

dom_2023 = pd.read_csv(os.path.join(pasta_dom, '2023_domicilios.csv'), sep=';', decimal=',')
dom_2024 = pd.read_csv(os.path.join(pasta_dom, '2024_domicilios.csv'), sep=';', decimal=',')
dom_2025 = pd.read_csv(os.path.join(pasta_dom, '2025_domicilios.csv'), sep=';', decimal=',')

# Mostra o tamanho de cada um (len = número de linhas, shape[1] = número de colunas)
for nome, df in [('ind_2023', ind_2023), ('ind_2024', ind_2024), ('ind_2025', ind_2025),
                 ('dom_2023', dom_2023), ('dom_2024', dom_2024), ('dom_2025', dom_2025)]:
    print(f'{nome}: {len(df)} linhas, {df.shape[1]} colunas')

# Guarda os totais pra conferir na seção 12
linhas_brutas_ind = len(ind_2023) + len(ind_2024) + len(ind_2025)
linhas_brutas_dom = len(dom_2023) + len(dom_2024) + len(dom_2025)
print('Total de pessoas:', linhas_brutas_ind, '| Total de domicílios:', linhas_brutas_dom)

ind_2023: 21271 linhas, 454 colunas
ind_2024: 21170 linhas, 439 colunas
ind_2025: 24535 linhas, 474 colunas
dom_2023: 23975 linhas, 264 colunas
dom_2024: 23856 linhas, 263 colunas
dom_2025: 27177 linhas, 271 colunas
Total de pessoas: 66976 | Total de domicílios: 75008


## 3. Dois sistemas de peso amostral (achado importante)

O módulo `domicilios` e o módulo `individuos` **têm cada um o seu próprio `PESO`**, e eles não são a mesma coisa. Testamos em 2023: 100% das 21.271 pessoas têm um peso de indivíduo diferente do peso da própria casa. As 200 colunas de peso réplica (`rep1` a `rep200`, usadas no erro padrão e no Rao-Scott) também são diferentes entre os módulos (13.798 de 21.271 pessoas já diferem no `rep1`).

Faz sentido: a chance de uma **casa** ser sorteada é diferente da chance de uma **pessoa** dentro dela ser a sorteada. Como o G1 e o G2 são respondidos por pessoa, **o peso certo pra qualquer conta com G1 ou G2 é o do indivíduo**, mesmo quando cruzado com uma variável da casa, como renda ou região. A unidade de análise continua sendo a pessoa.

Por isso o crosswalk tem `peso_amostral_domicilio` e `peso_amostral_individuo` como variáveis separadas, e nunca um `peso_amostral` genérico que poderia ser usado errado sem ninguém perceber.

## 4. Harmonizar os nomes das colunas

- O mesmo dado pode ter nomes diferentes em cada ano. Exemplo: a escolaridade se chama `GRAU_INSTRUCAO_2` em 2023 e `GRAU_INST_1` em 2024 e 2025.
- O **crosswalk** (`docs/lista_final_variaveis_projeto.csv`) é a tabela de tradução: pra cada variável, o nome final e o nome que ela tem em cada ano.
- Se um nome mudar, a correção é numa linha do CSV, não no código. A coluna `observacao` do crosswalk explica os códigos de cada variável.

In [3]:
# Lê a tabela de tradução dos nomes
crosswalk = pd.read_csv(os.path.join(raiz_projeto, 'docs', 'lista_final_variaveis_projeto.csv'))

# Mostra só as colunas que o código usa (a coluna observacao é pra gente ler)
crosswalk[['variavel_final', 'modulo', 'nome_2023', 'nome_2024', 'nome_2025']]

,variavel_final,modulo,nome_2023,nome_2024,nome_2025
0,chave_domicilio,domicilios+individuos,ID_DOMICILIO,ID_DOMICILIO,ID_DOMICILIO
1,peso_amostral_domicilio,domicilios,PESO,PESO,PESO
2,peso_amostral_individuo,individuos,PESO,PESO,PESO
3,renda_familiar,domicilios,RENDA_FAMILIAR_2,RENDA_FAMILIAR_2,RENDA_FAMILIAR_2
4,classe_economica,domicilios,CLASSE_2015,CLASSE_2015,CLASSE_2015
5,escolaridade,individuos,GRAU_INSTRUCAO_2,GRAU_INST_1,GRAU_INST_1
6,raca,individuos,RACA,RACA,RACA
7,sexo,individuos,SEXO,SEXO,SEXO
8,idade,individuos,IDADE,IDADE,IDADE
9,regiao,domicilios,COD_REGIAO_2,COD_REGIAO_2,COD_REGIAO_2


### 4.1 Padronizar os pesos réplica

- Em domicílios, 2023 vem com `rep1` a `rep200` (minúsculo) e 2024/2025 com `REP1` a `REP200` (maiúsculo). Empilhando assim, o pandas criaria 400 colunas, com metade vazia em cada ano.
- A função `padronizar_pesos_replica` deixa tudo minúsculo. Em indivíduos já vem minúsculo, então ela não muda nada lá (fica por segurança).
- **2023, indivíduos: as réplicas não vêm como peso.** No arquivo de 2023, `rep1` a `rep200` são **contagens** do sorteio do bootstrap: números inteiros de 0 a 5 que dizem quantas vezes a pessoa (junto com o conglomerado dela) foi sorteada em cada réplica. O peso de cada réplica é o `PESO` da pessoa vezes a contagem, e a célula faz essa conta. Em 2024 e 2025 as réplicas já vêm prontas, como peso: cada uma soma o total da população.
    - Como a gente descobriu (12/10): comparando as margens de erro com as tabelas oficiais da CETIC, no notebook 03. Usando as contagens direto, a margem de 2023 saía errada; com `PESO` × contagem, fica perto da oficial.
    - Os domicílios de 2023 também vêm com contagens, mas as réplicas da casa não vão para os arquivos finais, então a célula não mexe nelas.
- **Detalhe importante:** a função devolve um DataFrame **novo**. Por isso cada linha tem o nome dos dois lados do `=`. Sem o `dom_2024 =` da esquerda, o resultado se perderia.

In [4]:
# Cada linha: pega o DataFrame, padroniza e guarda de volta no mesmo nome
dom_2023 = padronizar_pesos_replica(dom_2023)
dom_2024 = padronizar_pesos_replica(dom_2024)
dom_2025 = padronizar_pesos_replica(dom_2025)

ind_2023 = padronizar_pesos_replica(ind_2023)
ind_2024 = padronizar_pesos_replica(ind_2024)
ind_2025 = padronizar_pesos_replica(ind_2025)

# 2023, indivíduos: rep1 a rep200 são CONTAGENS do bootstrap (0 a 5), não pesos.
# O peso de cada réplica = peso da pessoa x contagem. Em 2024 e 2025 já vem pronto
colunas_rep = [f'rep{i}' for i in range(1, 201)]          # ['rep1', 'rep2', ..., 'rep200']

# Pra ver a conta: 3 pessoas que foram sorteadas 2 vezes na réplica 1
exemplo = ind_2023[ind_2023['rep1'] == 2].index[:3]
print('2023, antes (contagens):')
print(ind_2023.loc[exemplo, ['PESO', 'rep1', 'rep2', 'rep3']].round(1), '\n')

# .mul(..., axis=0) multiplica cada LINHA pelo PESO daquela mesma pessoa
ind_2023[colunas_rep] = ind_2023[colunas_rep].mul(ind_2023['PESO'], axis=0)

print('2023, depois (pesos):')
print(ind_2023.loc[exemplo, ['PESO', 'rep1', 'rep2', 'rep3']].round(1))


2023, antes (contagens):
       PESO  rep1  rep2  rep3
221   256.6     2     3     1
222  1195.6     2     3     1
223   992.9     2     3     1 

2023, depois (pesos):
       PESO    rep1    rep2    rep3
221   256.6   513.1   769.7   256.6
222  1195.6  2391.2  3586.8  1195.6
223   992.9  1985.8  2978.6   992.9


### 4.2 Aplicar o rename

- `renomear_colunas` filtra o crosswalk pelo módulo, monta um dicionário `{nome no arquivo: nome final}` e renomeia.
- O filtro por módulo é necessário porque o mesmo `PESO` vira `peso_amostral_domicilio` num módulo e `peso_amostral_individuo` no outro.
- Colunas que não estão no crosswalk (`G1_A`, `ID_MORADOR`, `A9`...) continuam com o nome original.
- A linha `usuario_internet` da lista faz o `F_C3_IDADE` da CETIC virar `usuario_internet` nos 3 anos (1 = usou a internet nos últimos 3 meses e tem 16 anos ou mais). A seção 10 explica pra que ela serve.
- No fim, a célula confere se as colunas principais existem com o nome novo.

In [5]:
# Renomeia cada ano/módulo usando o crosswalk
dom_2023 = renomear_colunas(dom_2023, 2023, 'domicilios', crosswalk)
dom_2024 = renomear_colunas(dom_2024, 2024, 'domicilios', crosswalk)
dom_2025 = renomear_colunas(dom_2025, 2025, 'domicilios', crosswalk)

ind_2023 = renomear_colunas(ind_2023, 2023, 'individuos', crosswalk)
ind_2024 = renomear_colunas(ind_2024, 2024, 'individuos', crosswalk)
ind_2025 = renomear_colunas(ind_2025, 2025, 'individuos', crosswalk)

# Confere se as colunas principais existem com o nome novo
esperadas_dom = ['chave_domicilio', 'peso_amostral_domicilio', 'renda_familiar', 'regiao',
                 'tipo_conexao', 'velocidade_conexao', 'valor_conexao']
esperadas_ind = ['chave_domicilio', 'peso_amostral_individuo', 'raca', 'escolaridade',
                 'transacoes_financeiras', 'usuario_internet']

for nome, df, esperadas in [('dom_2023', dom_2023, esperadas_dom), ('dom_2024', dom_2024, esperadas_dom),
                            ('dom_2025', dom_2025, esperadas_dom), ('ind_2023', ind_2023, esperadas_ind),
                            ('ind_2024', ind_2024, esperadas_ind), ('ind_2025', ind_2025, esperadas_ind)]:
    faltando = [c for c in esperadas if c not in df.columns]   # lista das que não apareceram
    if faltando:
        print(nome, 'FALTANDO:', faltando)
    else:
        print(nome, 'ok')

dom_2023 ok
dom_2024 ok
dom_2025 ok
ind_2023 ok
ind_2024 ok
ind_2025 ok


## 5. Marcar o ano

- Depois de empilhar os 3 anos, a coluna `ano` é o único jeito de saber de onde veio cada linha.
- Aqui não precisa do `nome = ...`: dentro do loop, `df` aponta pro **mesmo** DataFrame que `dom_2023` (e assim por diante). Criar uma coluna com `df['ano'] = ...` muda esse DataFrame no lugar.
- Essa diferença entre "mudar no lugar" e "criar um novo" é uma das maiores fontes de erro no pandas, então vale prestar atenção nela.
- **Por que o `.copy()` antes do loop:** no pandas 3, o `read_csv` guarda cada coluna num pedaço separado da memória (um pedaço por coluna: de 263 a 474 por tabela). Criar uma coluna nova numa tabela assim fica lento, e o pandas avisava com um `PerformanceWarning: DataFrame is highly fragmented`. O `.copy()` faz uma cópia com as colunas juntas em poucos pedaços, e o aviso some. O resultado é o mesmo.
- Repare na diferença: o `.copy()` cria uma tabela **nova**, então precisa do `nome =` na frente. O `df['ano'] = ...` muda a tabela **no lugar**, então não precisa.

In [6]:
# Junta as colunas de cada tabela na memória (ver o texto acima).
# O .copy() cria uma tabela NOVA, por isso precisa do "nome =" na frente.
# a, b, c = x, y, z troca as três de uma vez
dom_2023, dom_2024, dom_2025 = dom_2023.copy(), dom_2024.copy(), dom_2025.copy()
ind_2023, ind_2024, ind_2025 = ind_2023.copy(), ind_2024.copy(), ind_2025.copy()

# Pra cada par (tabela, ano), cria a coluna 'ano' com o mesmo valor em todas as linhas
for df, ano in [(dom_2023, 2023), (dom_2024, 2024), (dom_2025, 2025),
                (ind_2023, 2023), (ind_2024, 2024), (ind_2025, 2025)]:
    df['ano'] = ano

## 6. Conferência: o que o questionário filtra

Cada pergunta tem um filtro no questionário, do tipo `### SOMENTE PARA ... ###`. O código 99 (não se aplica) quer dizer só que a pergunta não foi feita, e **quem** ficou de fora depende do filtro. Esta seção só mostra tabelas, não muda nenhum dado.

- **Velocidade (A8 em 2023/2024, A8A em 2025):** só é perguntada a quem tem banda larga fixa, ou seja, tipo de conexão (A7) igual a 2, 3, 4 ou 5. A primeira tabela mostra, pra cada tipo de conexão, quantas casas têm faixa de velocidade e quantas têm 99. É por isso que a seção 9 usa a A7 pra separar o 99.
- **Valor (A9_FAIXA):** as faixas de valor não têm a opção "não paga nada", que existe na pergunta original (A9 = 17 em 2023/2024, A9A = 0 em 2025). A segunda tabela mostra em qual faixa da A9_FAIXA essas casas caíram.

**O que os dados reais mostraram (conferido em 10/10)**
- Móvel (A7 = 6), não sabe (97), não respondeu (98) e sem internet (99) têm velocidade 99 nos 3 anos, como o filtro manda.
- A discada (A7 = 1) foge do filtro: em 2023 e 2024 essas casas têm faixa de velocidade (90 e 79 casas), e em 2025 têm 99. A seção 9 manda a discada pra NaN nos 3 anos, pra regra ficar igual.
- Quem não paga a internet caiu na **faixa 1** (até R$ 30) em 2023 e 2024, e aparece com um **código 0** em 2025, que nem está no dicionário. A seção 8 cria o código 0 = "não paga nada" nos 3 anos.

In [7]:
for dom, ano in [(dom_2023, 2023), (dom_2024, 2024), (dom_2025, 2025)]:
    print(f'================ {ano} ================')

    # Tabela 1: tipo de conexão (linhas) x "a velocidade é 99?" (colunas)
    velocidade_99 = dom['velocidade_conexao'] == 99             # True ou False pra cada casa
    tabela = pd.crosstab(dom['tipo_conexao'], velocidade_99)    # conta quantas casas em cada combinação
    tabela = tabela.rename(columns={False: 'tem faixa', True: 'velocidade = 99'})
    print('Tipo de conexão (A7) x velocidade:')
    print(tabela, '\n')

    # Tabela 2: onde caiu quem disse que não paga nada pela internet
    coluna_valor = 'A9A' if ano == 2025 else 'A9'               # a pergunta original muda de nome em 2025
    codigo_nao_paga = 0 if ano == 2025 else 17                  # e o código de "não paga nada" também
    if coluna_valor in dom.columns:
        nao_paga = dom[coluna_valor] == codigo_nao_paga
        print(f'Casas que não pagam nada ({nao_paga.sum()}), por faixa da A9_FAIXA:')
        print(dom.loc[nao_paga, 'valor_conexao'].value_counts(dropna=False), '\n')
    else:
        print(f'Coluna {coluna_valor} não encontrada, pulando a tabela 2\n')

================ 2023 ================
Tipo de conexão (A7) x velocidade:
velocidade_conexao  tem faixa  velocidade = 99
tipo_conexao                                  
1                          90                0
2                         200                0
3                       10652                0
4                         498                0
5                         764                0
6                           0             3259
97                          0             2482
98                          0              105
99                          0             5925 

Casas que não pagam nada (450), por faixa da A9_FAIXA:
valor_conexao
1    450
Name: count, dtype: int64 

================ 2024 ================
Tipo de conexão (A7) x velocidade:
velocidade_conexao  tem faixa  velocidade = 99
tipo_conexao                                  
1                          79                0
2                         162                0
3                       11242          

## 7. Recodificar G1 e G2

- **G1_X** (usou a internet pro serviço X nos últimos 12 meses?) já vem com 0 = não e 1 = sim. Só os códigos 97, 98 e 99 viram NaN.
- **G2_X** (fez o serviço inteiro pela internet?) vira 1 = fez tudo online, e 0 = precisou ir ao posto (código 2) ou só procurou informação (código 3). Os códigos 97, 98 e 99 viram NaN. A função `recodificar_g2` está em `src/harmonizacao.py`.
- A área H (Justiça) só existe em 2025.
- **`.map` ou `.replace`?** O `.map` traduz tudo pelo dicionário, e o que não estiver lá vira NaN. Serve pra recodificação completa, como a do G2. O `.replace` troca só os valores listados e mantém o resto. Serve pro G1, onde 0 e 1 ficam como estão.
- Usamos `np.nan`, e não `None`, pra coluna continuar numérica. No pandas 3, trocar por `None` deixa a coluna como texto.

In [8]:
# Dicionário com os códigos que viram NaN no G1
nao_resposta_g1 = {97: np.nan, 98: np.nan, 99: np.nan}

for df, ano in [(ind_2023, 2023), (ind_2024, 2024), (ind_2025, 2025)]:
    # Lista das áreas desse ano: A a G em todos os anos, mais H em 2025
    areas = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
    if ano == 2025:
        areas = areas + ['H']

    for area in areas:
        coluna_g1 = f'G1_{area}'    # ex: 'G1_B'
        coluna_g2 = f'G2_{area}'    # ex: 'G2_B'
        df[coluna_g2] = recodificar_g2(df[coluna_g2])            # 1/2/3 viram 1/0/0 e 97/98/99 viram NaN
        df[coluna_g1] = df[coluna_g1].replace(nao_resposta_g1)   # 97/98/99 viram NaN; 0 e 1 ficam

# Confere em 2024, área B (saúde): só pode sobrar 0, 1 e NaN
print('G1_B em 2024:', ind_2024['G1_B'].value_counts(dropna=False).to_dict())
print('G2_B em 2024:', ind_2024['G2_B'].value_counts(dropna=False).to_dict())

G1_B em 2024: {0.0: 11554, nan: 5545, 1.0: 4071}
G2_B em 2024: {nan: 17115, 0.0: 3050, 1.0: 1005}


## 8. Limpar os códigos de não resposta

Os códigos vêm dos dicionários oficiais (`docs/fontes_cetic/dicionarios/`) e são iguais nos 3 anos:

| Coluna | Módulo | Códigos que ficam | Viram NaN |
| --- | --- | --- | --- |
| `raca` | indivíduos | 1 a 5 | 98 (não respondeu) |
| `transacoes_financeiras` | indivíduos | 0 e 1 | 97, 98, 99 |
| `renda_familiar` | domicílios | 1 a 6 e 7 (não tem renda) | 97, 98 |
| `valor_conexao` | domicílios | 1 a 10, mais o 0 = não paga nada | 97, 98, 99 |

- O 7 da renda ("não tem renda") é uma resposta de verdade, então fica. Como usar esse grupo na análise é uma decisão do grupo, registrada no caderno de bordo.
- No valor da conexão, o 99 é a casa sem internet, então vira NaN (o valor não se aplica).
- **"Não paga nada" vira o código 0 nos 3 anos.** A seção 6 mostrou que, em 2023 e 2024, a CETIC colocou essas casas na faixa 1 (até R$ 30), e em 2025 usou um código 0 que não está no dicionário. Pra ficar igual, em 2023 e 2024 a célula usa a pergunta original (A9 = 17) pra tirar essas casas da faixa 1 e passar pro 0.
- A célula mostra a contagem de 2024 antes e depois, pra ver os códigos sumindo e virando NaN.

In [9]:
# ANTES: contagem de 2024 com os códigos de não resposta ainda lá
print('ANTES (2024)')
print('  renda_familiar:', dom_2024['renda_familiar'].value_counts(dropna=False).sort_index().to_dict())
print('  transacoes_financeiras:', ind_2024['transacoes_financeiras'].value_counts(dropna=False).sort_index().to_dict())

# Indivíduos: raça e transações financeiras
for df in [ind_2023, ind_2024, ind_2025]:
    df['raca'] = df['raca'].replace({98: np.nan})
    df['transacoes_financeiras'] = df['transacoes_financeiras'].replace({97: np.nan, 98: np.nan, 99: np.nan})

# Domicílios: renda e valor da conexão
for df in [dom_2023, dom_2024, dom_2025]:
    df['renda_familiar'] = df['renda_familiar'].replace({97: np.nan, 98: np.nan})
    df['valor_conexao'] = df['valor_conexao'].replace({97: np.nan, 98: np.nan, 99: np.nan})

# Valor da conexão: "não paga nada" vira o código 0 nos 3 anos.
# Em 2023 e 2024 essas casas estão na faixa 1; a pergunta original (A9) tem 17 = não paga nada
for df in [dom_2023, dom_2024]:
    nao_paga = df['A9'] == 17                     # True pras casas que não pagam
    df.loc[nao_paga, 'valor_conexao'] = 0         # .loc[linhas, coluna] = troca só nessas linhas
# Em 2025 elas já vêm com 0 na A9_FAIXA, então não precisa mexer

# DEPOIS: os códigos 97/98/99 somem e o NaN passa a ter a soma deles
print('DEPOIS (2024)')
print('  renda_familiar:', dom_2024['renda_familiar'].value_counts(dropna=False).sort_index().to_dict())
print('  transacoes_financeiras:', ind_2024['transacoes_financeiras'].value_counts(dropna=False).sort_index().to_dict())

# Quantas casas ficaram com valor 0 (não paga nada) em cada ano
for df, ano in [(dom_2023, 2023), (dom_2024, 2024), (dom_2025, 2025)]:
    print(f'  {ano}: {(df["valor_conexao"] == 0).sum()} casas não pagam nada pela internet')

ANTES (2024)
  renda_familiar: {1: 9200, 2: 6576, 3: 3308, 4: 1761, 5: 814, 6: 271, 7: 365, 97: 922, 98: 639}
  transacoes_financeiras: {0: 8213, 1: 8561, 97: 7, 98: 9, 99: 4380}
DEPOIS (2024)
  renda_familiar: {1.0: 9200, 2.0: 6576, 3.0: 3308, 4.0: 1761, 5.0: 814, 6.0: 271, 7.0: 365, nan: 1561}
  transacoes_financeiras: {0.0: 8213, 1.0: 8561, nan: 4396}
  2023: 450 casas não pagam nada pela internet
  2024: 373 casas não pagam nada pela internet
  2025: 351 casas não pagam nada pela internet


## 9. Velocidade e tipo de conexão

A pergunta de velocidade só é feita a quem tem banda larga fixa:

```
Domicílio
├─ tem internet? não  →  A7 = 99 e velocidade = 99   (sem internet)
└─ sim  →  A7 = tipo de conexão
     ├─ 2 a 5 (DSL, cabo/fibra, rádio, satélite)  →  velocidade = faixa
     ├─ 6 (rede móvel 3G/4G/5G)                   →  velocidade = 99
     └─ 1 (discada)                               →  velocidade = 99
```

Por isso a função `recodificar_velocidade` usa a velocidade **e** o tipo de conexão:

| Tipo de conexão (A7) | `velocidade_conexao` final |
| --- | --- |
| 2 a 5 (banda larga fixa) | `baixa` (até 10 Mbps), `media` (11 a 50), `alta` (51 ou mais) |
| 6 (rede móvel) | `movel` |
| 99 (sem internet) | `sem internet` |
| 1 (discada), 97, 98 | NaN |

- A discada vira NaN mesmo quando tem faixa de velocidade (aconteceu em 2023 e 2024, ver seção 6), pra regra ficar igual nos 3 anos.
- O maior grupo de NaN é o A7 = 97: casas com internet que não souberam dizer o tipo de conexão.
- **Mudança de 10/10:** a regra antiga transformava todo 99 em "sem conexão". Isso colocava as casas que só usam internet móvel junto com as que não têm internet. A mudança está registrada no caderno de bordo.
- A coluna `tipo_conexao` só serve pra essa conta e não vai pros arquivos finais.
- Em 2025 a A7 também é feita pra casas com Wi-Fi ou com morador que acessa a internet em casa, então o "sem internet" de 2025 é um pouco mais restrito. Vale uma nota no método.

In [10]:
# Recodifica a velocidade de cada ano usando também o tipo de conexão
for df, ano in [(dom_2023, 2023), (dom_2024, 2024), (dom_2025, 2025)]:
    df['velocidade_conexao'] = recodificar_velocidade(df['velocidade_conexao'], df['tipo_conexao'], ano)

# Confere: quantas casas em cada categoria, por ano
for df, ano in [(dom_2023, 2023), (dom_2024, 2024), (dom_2025, 2025)]:
    print(ano, df['velocidade_conexao'].value_counts(dropna=False).to_dict())

2023 {nan: 5987, 'sem internet': 5925, 'alta': 4332, 'movel': 3259, 'baixa': 3084, 'media': 1388}
2024 {nan: 6885, 'sem internet': 5242, 'alta': 4801, 'baixa': 3166, 'movel': 2738, 'media': 1024}
2025 {nan: 8200, 'alta': 7225, 'sem internet': 4524, 'movel': 3131, 'baixa': 3071, 'media': 1026}


## 10. Escolher as colunas

Os arquivos brutos têm muitas colunas. A gente fica só com as que a pesquisa usa:

- `cols_dom`: as colunas do arquivo de domicílios, com o peso do domicílio.
- `cols_dom_para_merge`: as mesmas colunas **sem** o peso. São as que vão ser levadas pra cada pessoa na seção 11. Assim o `dados_harmonizados` nunca fica com dois pesos.
- `cols_ind_fixas`: as colunas de indivíduos que existem nos 3 anos. As colunas `G1_` e `G2_` são escolhidas na seção 11, porque 2025 tem a área H.
- `usuario_internet` entra nas colunas de indivíduos. É o universo das tabelas oficiais do módulo G: 1 = usou a internet nos últimos 3 meses (pergunta C3) e tem 16 anos ou mais. O questionário aplicou o módulo G também a quem disse "não" na C3 e "sim" na J3 (internet no celular nos últimos 3 meses): umas mil pessoas por ano, que ficam fora das tabelas oficiais. **O notebook 01 não tira ninguém:** os arquivos guardam todo mundo, e os notebooks 02 e 03 filtram `usuario_internet == 1` (decisão de 12/10, no caderno de bordo).
- `cols_replica_individuo`: as chaves, o peso e as 200 réplicas do indivíduo. Elas vão pra um arquivo separado porque são pesadas: juntas com o resto, deixavam os CSVs com mais de 90 MB.

In [11]:
# Domicílios: o que vai pro arquivo de domicílios (com o peso da casa)
cols_dom = ['chave_domicilio', 'ano', 'peso_amostral_domicilio', 'renda_familiar', 'classe_economica',
            'regiao', 'area_urbano_rural', 'velocidade_conexao', 'valor_conexao']

# Domicílios: o que vai pra cada pessoa no merge (sem o peso da casa)
cols_dom_para_merge = ['chave_domicilio', 'ano', 'renda_familiar', 'classe_economica',
                       'regiao', 'area_urbano_rural', 'velocidade_conexao', 'valor_conexao']

# Indivíduos: colunas que existem nos 3 anos
cols_ind_fixas = ['chave_domicilio', 'ID_MORADOR', 'ano', 'idade', 'sexo', 'raca', 'escolaridade',
                  'g1_agreg', 'transacoes_financeiras', 'usuario_internet',
                  'peso_amostral_individuo']

# Réplicas: chaves + peso + rep1 a rep200
# [f'rep{i}' for i in range(1, 201)] é uma "list comprehension": um loop numa linha só
# que monta a lista ['rep1', 'rep2', ..., 'rep200']
cols_replica_individuo = ['chave_domicilio', 'ID_MORADOR', 'ano', 'peso_amostral_individuo'] + \
                         [f'rep{i}' for i in range(1, 201)]

## 11. Empilhar os anos e juntar os módulos

- `pd.concat` empilha as linhas dos 3 anos. Ele alinha as colunas **pelo nome**, por isso os nomes precisavam ficar iguais na seção 4. O `ignore_index=True` numera as linhas de novo, de 0 até o fim.
- G1_H e G2_H só existem em 2025, então ficam vazias (NaN) em 2023 e 2024 depois de empilhar.
- `merge` leva os dados da casa (renda, região...) pra cada pessoa, usando a chave `chave_domicilio` + `ano`:
    - `how='left'`: mantém todas as pessoas, mesmo que alguma não ache a casa.
    - `validate='many_to_one'`: várias pessoas podem apontar pra mesma casa, mas cada casa tem que aparecer uma vez só. Se alguma estiver duplicada, o pandas dá erro em vez de duplicar pessoas sem avisar.
    - `indicator=True`: cria uma coluna `_merge` dizendo se cada pessoa achou a casa (`both`). A célula conta quem não achou e depois apaga essa coluna.

In [12]:
# Domicílios: empilha os 3 anos, só com as colunas escolhidas
domicilios_harmonizado = pd.concat(
    [dom_2023[cols_dom], dom_2024[cols_dom], dom_2025[cols_dom]],
    ignore_index=True,
)

# Indivíduos: as colunas G1_/G2_ mudam de um ano pro outro (2025 tem a área H)
partes_individuos = []    # lista vazia que vai receber um pedaço por ano
partes_replica = []
for df in [ind_2023, ind_2024, ind_2025]:
    colunas_g1_g2 = [c for c in df.columns if c.startswith('G1_') or c.startswith('G2_')]
    partes_individuos.append(df[cols_ind_fixas + colunas_g1_g2])   # o + junta as duas listas de colunas
    partes_replica.append(df[cols_replica_individuo])

individuos_harmonizado = pd.concat(partes_individuos, ignore_index=True)
individuos_pesos_replica = pd.concat(partes_replica, ignore_index=True)

# Junta cada pessoa com os dados da casa dela
dados_harmonizados = individuos_harmonizado.merge(
    domicilios_harmonizado[cols_dom_para_merge],
    on=['chave_domicilio', 'ano'],
    how='left',
    validate='many_to_one',
    indicator=True,
)

# Conta quantas pessoas não acharam a casa (o esperado é 0) e apaga a coluna auxiliar
pessoas_sem_casa = (dados_harmonizados['_merge'] != 'both').sum()
dados_harmonizados = dados_harmonizados.drop(columns='_merge')

print('domicilios_harmonizado:', domicilios_harmonizado.shape)
print('individuos_harmonizado:', individuos_harmonizado.shape)
print('dados_harmonizados:', dados_harmonizados.shape)
print('individuos_pesos_replica:', individuos_pesos_replica.shape)
print('Pessoas sem casa no merge:', pessoas_sem_casa)

domicilios_harmonizado: (75008, 9)
individuos_harmonizado: (66976, 27)
dados_harmonizados: (66976, 33)
individuos_pesos_replica: (66976, 204)
Pessoas sem casa no merge: 0


## 12. Testes de integridade

Antes de salvar, o notebook confere se nada deu errado. São duas partes:

1. **Estrutura:** nenhuma linha perdida ou duplicada, casas sem repetição, todo mundo com casa, pesos preenchidos e réplicas com cara de peso (em cada ano, a soma de cada réplica fica perto da soma do peso; se as contagens de 2023 não tivessem virado peso, a soma daria umas 21 mil em vez de uns 186 milhões). Cada `assert` é uma pergunta de sim ou não. Se a resposta for "não", o notebook para com uma mensagem dizendo o que falhou, e nada é salvo com erro.
2. **Valores:** cada coluna só pode ter os códigos esperados. Aqui o notebook só avisa e não para, porque um código inesperado precisa ser investigado no dicionário antes de decidir o que fazer com ele.

In [13]:
# ---------- 1. Estrutura ----------

# Nenhuma linha perdida nem duplicada ao empilhar
assert len(domicilios_harmonizado) == linhas_brutas_dom, 'Domicílios: o número de linhas mudou'
assert len(individuos_harmonizado) == linhas_brutas_ind, 'Indivíduos: o número de linhas mudou'

# O merge não pode criar nem apagar pessoas
assert len(dados_harmonizados) == len(individuos_harmonizado), 'O merge mudou o número de pessoas'

# Cada casa aparece uma vez só em cada ano
assert not domicilios_harmonizado.duplicated(['chave_domicilio', 'ano']).any(), 'Tem casa duplicada'

# Toda pessoa achou a casa dela
assert pessoas_sem_casa == 0, f'{pessoas_sem_casa} pessoas não acharam a casa no merge'

# O arquivo de réplicas tem as mesmas pessoas, na mesma ordem
chaves_pessoa = ['chave_domicilio', 'ID_MORADOR', 'ano']
assert individuos_pesos_replica[chaves_pessoa].equals(individuos_harmonizado[chaves_pessoa]), \
    'O arquivo de réplicas não tem as mesmas pessoas na mesma ordem'

# Pesos preenchidos e positivos
assert dados_harmonizados['peso_amostral_individuo'].notna().all(), 'Peso do indivíduo com valor faltando'
assert (dados_harmonizados['peso_amostral_individuo'] > 0).all(), 'Peso do indivíduo zero ou negativo'
colunas_rep = [f'rep{i}' for i in range(1, 201)]
assert individuos_pesos_replica[colunas_rep].notna().all().all(), 'Peso réplica com valor faltando'

# As réplicas são pesos: em cada ano, a soma de cada réplica fica perto da soma do peso
# (2024 e 2025: igual; 2023: entre 0,93 e 1,10. Com as contagens, daria perto de 0)
soma_peso = individuos_pesos_replica.groupby('ano')['peso_amostral_individuo'].sum()     # uma soma por ano
soma_reps = individuos_pesos_replica.groupby('ano')[colunas_rep].sum()                   # uma linha por ano, uma coluna por réplica
razao_reps = soma_reps.div(soma_peso, axis=0)                                            # divide cada linha pela soma do peso do ano
assert razao_reps.min().min() > 0.5 and razao_reps.max().max() < 1.5, 'Alguma réplica não parece peso'

# Só o peso do indivíduo pode estar no arquivo principal
assert 'peso_amostral_domicilio' not in dados_harmonizados.columns, 'O peso do domicílio foi parar no arquivo principal'

print('Estrutura: tudo certo')
print(f'  {len(domicilios_harmonizado)} domicílios e {len(individuos_harmonizado)} pessoas')


# ---------- 2. Valores ----------

def conferir_valores(df, coluna, permitidos):
    # Avisa se a coluna tem algum valor fora da lista de permitidos (NaN é aceito)
    valores = set(df[coluna].dropna().unique())    # valores diferentes que aparecem na coluna
    fora = valores - set(permitidos)                # os que não estão na lista de permitidos
    if fora:
        print(f'  ATENÇÃO {coluna}: valores inesperados {sorted(fora)}')
    else:
        print(f'  ok  {coluna}')


print('\nValores das colunas de indivíduos:')
for area in ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']:
    conferir_valores(dados_harmonizados, f'G1_{area}', [0, 1])
    conferir_valores(dados_harmonizados, f'G2_{area}', [0, 1])
conferir_valores(dados_harmonizados, 'raca', [1, 2, 3, 4, 5])
conferir_valores(dados_harmonizados, 'sexo', [1, 2])
conferir_valores(dados_harmonizados, 'escolaridade', [1, 2, 3, 4])
conferir_valores(dados_harmonizados, 'transacoes_financeiras', [0, 1])
conferir_valores(dados_harmonizados, 'g1_agreg', [0, 1])
conferir_valores(dados_harmonizados, 'usuario_internet', [0, 1])

print('\nValores das colunas de domicílios:')
conferir_valores(domicilios_harmonizado, 'renda_familiar', [1, 2, 3, 4, 5, 6, 7])
conferir_valores(domicilios_harmonizado, 'classe_economica', [1, 2, 3, 4])
conferir_valores(domicilios_harmonizado, 'regiao', [1, 2, 3, 4, 5])
conferir_valores(domicilios_harmonizado, 'area_urbano_rural', [1, 2])
conferir_valores(domicilios_harmonizado, 'valor_conexao', list(range(0, 11)))   # 0 = não paga nada
conferir_valores(domicilios_harmonizado, 'velocidade_conexao', ['baixa', 'media', 'alta', 'movel', 'sem internet'])

Estrutura: tudo certo
  75008 domicílios e 66976 pessoas

Valores das colunas de indivíduos:
  ok  G1_A
  ok  G2_A
  ok  G1_B
  ok  G2_B
  ok  G1_C
  ok  G2_C
  ok  G1_D
  ok  G2_D
  ok  G1_E
  ok  G2_E
  ok  G1_F
  ok  G2_F
  ok  G1_G
  ok  G2_G
  ok  G1_H
  ok  G2_H
  ok  raca
  ok  sexo
  ok  escolaridade
  ok  transacoes_financeiras
  ok  g1_agreg
  ok  usuario_internet

Valores das colunas de domicílios:
  ok  renda_familiar
  ok  classe_economica
  ok  regiao
  ok  area_urbano_rural
  ok  valor_conexao
  ok  velocidade_conexao


## 13. Salvar em data/processed/

- São 4 arquivos (ver `data/processed/README.md`). Nenhum vai pro Git: qualquer pessoa do grupo gera de novo rodando este notebook.
- `index=False`: sem isso, o CSV ganharia uma coluna extra com o número de cada linha.
- O arquivo de réplicas tem cerca de 170 MB e demora um pouco pra salvar. Era 133 MB até 12/10: as réplicas de 2023 eram contagens (0 a 5) e viraram pesos, com casas decimais.

In [14]:
pasta_saida = os.path.join(raiz_projeto, 'data', 'processed')
os.makedirs(pasta_saida, exist_ok=True)    # cria a pasta se ela ainda não existir

# Nome do arquivo -> tabela que vai nele
arquivos = {
    'domicilios_harmonizado.csv': domicilios_harmonizado,
    'individuos_harmonizado.csv': individuos_harmonizado,
    'dados_harmonizados.csv': dados_harmonizados,
    'individuos_pesos_replica.csv': individuos_pesos_replica,
}

for nome_arquivo, tabela in arquivos.items():
    caminho = os.path.join(pasta_saida, nome_arquivo)
    tabela.to_csv(caminho, index=False)
    tamanho_mb = os.path.getsize(caminho) / 1_000_000
    print(f'{nome_arquivo}: {tabela.shape[0]} linhas x {tabela.shape[1]} colunas, {tamanho_mb:.1f} MB')

domicilios_harmonizado.csv: 75008 linhas x 9 colunas, 2.9 MB
individuos_harmonizado.csv: 66976 linhas x 27 colunas, 5.6 MB
dados_harmonizados.csv: 66976 linhas x 33 colunas, 6.9 MB
individuos_pesos_replica.csv: 66976 linhas x 204 colunas, 169.2 MB
